# Official GENCO checkpoint for the certified AC-RSS entry

Runs your experiments E3 and E4 with the **official** `gridfm/genco-pf-datakit` case118 checkpoint instead of the self-trained one, then compares the numbers.

* Runtime: **CPU is fine** (Runtime → Change runtime type → CPU). About 1–1.5 h in total.
* Before running, add `tools/prepare_official_checkpoint.py` and `tools/compare_genco.py` to your GitHub repo.
* Run the cells in order. Stop at cell 6 if the official model is not clearly more accurate.

In [ ]:
# 1. Get your code
REPO = "https://github.com/abdillah8384/genco-certified-ac-rss.git"
!git clone -q $REPO /content/genco 2>/dev/null || (cd /content/genco && git pull -q)
%cd /content/genco
!python --version
import os
assert os.path.exists("tools/prepare_official_checkpoint.py"), "Upload the two tools/ files to the repo first."


In [ ]:
# 2. Install the same stack as setup.sh (CPU build, no venv)
!pip install -q "torch==2.12.*" --index-url https://download.pytorch.org/whl/cpu
TV = !python -c "import torch; print(torch.__version__.split('+')[0])"
TV = TV[-1]; print("torch", TV)
!pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TV}+cpu.html
!pip install -q -r requirements-full.txt huggingface_hub safetensors
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"


In [ ]:
# 3. See what the official repository contains (nothing is downloaded yet)
from huggingface_hub import list_repo_files, hf_hub_download
REPO_ID = "gridfm/genco-pf-datakit"
files = list_repo_files(REPO_ID)
print(len(files), "files in", REPO_ID)
for f in files:
    if "118" in f or f.lower().endswith((".md", ".yaml", ".yml", ".json")):
        print("  ", f)
if "README.md" in files:
    print("\n----- model card (first 80 lines): check the graphkit version/branch it asks for -----")
    print("\n".join(open(hf_hub_download(REPO_ID, "README.md")).read().splitlines()[:80]))


In [ ]:
# 4. Download only the case118 files (plus configs and the model card)
from huggingface_hub import snapshot_download
snapshot_download(REPO_ID, local_dir="genco-pf-datakit",
                  allow_patterns=["*118*", "*118*/**", "*.md", "*.yaml", "*.yml"])
!find genco-pf-datakit -path '*118*' -type f | head -50


In [ ]:
# 5. Convert to the layout your pipeline loads, and check it loads strictly.
#    Set SIZE/SEED to tokens that appear in the paths printed above ("" to ignore).
SIZE, SEED = "small", "0"
OUT = "weights/genco_official_case118"
!python tools/prepare_official_checkpoint.py --download-dir genco-pf-datakit --grid 118 \
      --size "$SIZE" --seed "$SEED" --out $OUT
# Several candidates listed?  add  --model-file <path> --config-file <path> --stats-file <path>
# No config found?  run:  !git clone -q https://github.com/gridfm/gridfm-graphkit /content/gk
#                   then rerun the line above with  --search-dir /content/gk


In [ ]:
# 6. Sanity check on 300 fresh exact-AC scenarios of your RSS region (a few minutes).
#    The official row should have clearly lower errors. If not: STOP, the setup does not match.
!python tools/compare_genco.py \
   --run weights/genco_pf_case118 configs/genco_pf_case118.yaml \
   --run $OUT $OUT/config.yaml


In [ ]:
# 7. Re-run E3 (accuracy, PTR, conformal) and E4 (AC-corrected RSS) with the official model.
#    E1/E5 do not use GENCO, so their results are kept. The self-trained results are backed up first.
!rm -rf results_selftrained && cp -r results results_selftrained
!python genco_ac_rssr_challenge.py --steps e3,e4,fig,numbers --genco-run $OUT --genco-config $OUT/config.yaml
!rm -rf results_official && cp -r results results_official


In [ ]:
# 8. Side-by-side numbers for the post
import json
a = json.load(open("results_selftrained/numbers.json")); b = json.load(open("results_official/numbers.json"))
keys = ["genco_params", "load_mae_iid", "load_mae_screen", "load_mae_unseen", "vm_err_screen",
        "raw_mae_test", "ptr_mae_test", "ptr_mae_red_pct", "traj_only_mae_red_pct",
        "clear_plain_005", "clear_ptr_005", "miss_ptr_005", "clear_ptr_001", "miss_ptr_001",
        "clear_unseen_ptr_005", "miss_unseen_ptr_005", "clear_unseen_ptr_001", "genco_ms", "ac_ms"]
keys += sorted(k for k in b if k.startswith(("e4", "dc_width")) and k not in keys)
fmt = lambda v: f"{v:14.4f}" if isinstance(v, float) else f"{str(v):>14s}"
print(f"{'metric':32s} {'self-trained':>14s} {'official':>14s}")
for k in keys:
    if k in a or k in b:
        print(f"{k:32s} {fmt(a.get(k, '-'))} {fmt(b.get(k, '-'))}")


In [ ]:
# 9. Download everything (results + converted checkpoint) to commit to your repo
!zip -qr official_run.zip results_official results_selftrained $OUT
from google.colab import files
files.download("official_run.zip")
